# Real-data GIS export and evidence workbench

**Question:** What survives an explicit, attributed transfer into two browser GIS apps?

Run all cells to inspect real **Fresno 2020 Census whole-tract** population and export a deterministic small selection. The complete bundled input contains **166 tracts / 764468 people**; these are whole intersecting tracts, not a clipped AOI headcount. The default exports the first 12 source IDs meeting the declared threshold. It is never relabeled as another chapter's data.

This notebook calls no models/services and needs no native GIS. Its [original standard-library helper](atlas_gis_handoff.py) validates structural 2D WGS84 geometry, declared attribution, units, IDs and finite values; it cannot certify topology or source authenticity. Download it with this notebook for use outside Atlas and keep the declared relative inputs. [Cross-tool and optional Ollama guide](browser-gis.ipynb).


In [1]:
import json
import sys
from pathlib import Path
from IPython.display import display, FileLink
from atlas_gis_handoff import load_input, census_metadata, summarize, write_handoff

# Canonical files, published/staged with this notebook; no network fetching.
manifest_path = Path('courses/onehealth/data/manifest.json')
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
metadata = census_metadata(manifest)
input_path = Path('courses/onehealth/data/tracts.geojson')
id_field = 'GEOID'
pathways = json.loads(Path('assets/browser-gis-pathways.json').read_text(encoding='utf-8'))
print('Subject pathways available:', len(pathways['pathways']))


Subject pathways available: 42


## Choose an input honestly

For your own **already geographic, approved** file, upload it and an attributed metadata JSON through Lab's file browser, then explicitly replace `input_path`, `metadata` and `id_field` above. The metadata must declare `dataset_id`, `title`, `observation_status`, `observation_time` (or explicitly unknown), `input_crs`, longitude/latitude `coordinate_order`, `field_units`, `method`, `missingness_policy`, `limitations` and separate `sources` records (`source_id`, `source_url`, `attribution`, `license_statement`, `license_evidence`). Copy actual source terms/metadata, including NC-SA, per-source limits, date and method; do not reuse the Census declaration. EPSG:4326 additionally needs documented `axis_order_evidence`. Original collection metadata, source properties and geometry survive in the package.

These fresh-input instructions exclude app-returned Atlas packages: retain their protected provenance and use the separate returned-artifact inspection recipe below; do not strip reserved fields or silently restore IDs.

An existing Map and Scene Atlas source-sample export is eligible after this same metadata review. Choose the intended subject field below; never use the Fresno population field as a substitute for cholera, reef, climate or clinical observations. Unknown/projected CRS, 3D coordinates, null geometry, malformed or antimeridian/polar polygons stop this route with an actionable error. Preserve unmapped records as a table and follow the source workflow; no silent repair, geocoding, reprojection or dimensional reduction occurs.


In [2]:
collection, input_record = load_input(input_path, metadata, id_field=id_field)
input_summary = summarize(collection, metadata['field_units'])
print('Actual input SHA-256:', input_record['sha256'])
print('Input feature count:', input_summary['feature_count'])
print('Declared measurements:', input_summary['measurements'])
if metadata['dataset_id'] == 'fresno-census-2020-tracts':
    assert input_summary['feature_count'] == 166
    assert input_summary['measurements']['POP100']['numeric_sum'] == 764468
print('Bounded preview:', json.dumps(input_summary['preview'], indent=2))


Actual input SHA-256: 08c2bf5667c70c7755aa8e64b37790b8be8cf22f09d96356a300e2ec85cdb079
Input feature count: 166
Declared measurements: {'POP100': {'absent': 0, 'null': 0, 'empty_string': 0, 'zero': 0, 'false': 0, 'numeric_count': 166, 'nonnumeric_count': 0, 'numeric_sum': 764468, 'units': 'people; whole 2020 Census tract'}}
Bounded preview: [
  {
    "id": "06019000100",
    "properties": {
      "GEOID": "06019000100",
      "STATE": "06",
      "COUNTY": "019",
      "TRACT": "000100",
      "NAME": "Census Tract 1",
      "POP100": 3840,
      "HU100": 773,
      "AREALAND": 843445,
      "AREAWATER": 0,
      "CENTLAT": "+36.7369244",
      "CENTLON": "-119.7927732"
    }
  },
  {
    "id": "06019000200",
    "properties": {
      "GEOID": "06019000200",
      "STATE": "06",
      "COUNTY": "019",
      "TRACT": "000200",
      "NAME": "Census Tract 2",
      "POP100": 3251,
      "HU100": 863,
      "AREALAND": 1988144,
      "AREAWATER": 0,
      "CENTLAT": "+36.7354931",
      "

## Change a parameter, predict, then export

Change `minimum_value` or `selection_limit` and predict which **actual IDs** change. Selection retains source order and does not clip geometry. Null, absent, zero, false and empty string are accounted separately. Empty selection is valid. For another dataset, set its actual field/units and appropriate selection rule explicitly.

Exports are under this notebook's `exports/` directory in Lab. Identical reruns retain identical files; changed results need a **new `output_stem`** so valuable previous work is preserved. The package stores exact selected/excluded IDs, input byte SHA, original metadata, source terms, measurement summary and output SHA. The helper binds the normalized loaded collection to its receipt to catch accidental edits; it does not authenticate the source. Its `max_bytes` limit covers the combined UTF-8 GeoJSON, method JSON and review Markdown; complete provenance is retained or the package fails before writing.


In [3]:
measurement_field = 'POP100'
minimum_value = 0
selection_limit = 12
output_stem = 'fresno-2020-first12'
assert measurement_field in metadata['field_units'], 'Choose a declared measurement and units.'
assert type(selection_limit) is int and 0 <= selection_limit <= 100, 'Use a bounded explicit selection.'
eligible = [f for f in collection['features']
            if type(f['properties'].get(measurement_field)) in (int, float)
            and f['properties'][measurement_field] >= minimum_value]
selected_ids = [f['id'] for f in eligible[:selection_limit]]
run_metadata = dict(metadata)
run_metadata['selection_parameters'] = {'measurement_field': measurement_field,
    'minimum_value': minimum_value, 'selection_limit': selection_limit,
    'rule': 'numeric value >= minimum; first eligible source IDs in source order'}
package = write_handoff(collection, run_metadata, input_record, Path('exports'),
                        output_stem, selected_ids=selected_ids)
print('Export selection:', package['method']['selection'])
print('Selected measurements:', package['method']['summary']['measurements'])
if sys.platform == 'emscripten':
    for artifact in package['paths'].values():
        display(FileLink(str(artifact)))
else:
    print('Artifact names:', ', '.join(artifact.as_posix() for artifact in package['paths'].values()))
    print('Run this notebook in Lab to create and download these artifacts; saved Book output does not publish temporary exports.')
print(package['review'])


Export selection: {'input_count': 166, 'output_count': 12, 'selected_ids': ['06019000100', '06019000200', '06019000300', '06019000400', '06019000501', '06019000502', '06019000601', '06019000602', '06019000701', '06019000702', '06019000901', '06019000902'], 'excluded_ids': ['06019001000', '06019001100', '06019001201', '06019001202', '06019001301', '06019001303', '06019001304', '06019001407', '06019001408', '06019001409', '06019001411', '06019001412', '06019001413', '06019001415', '06019001416', '06019001417', '06019001418', '06019001500', '06019001600', '06019001700', '06019001800', '06019002000', '06019002100', '06019002200', '06019002300', '06019002400', '06019002501', '06019002502', '06019002601', '06019002602', '06019002701', '06019002702', '06019002800', '06019002903', '06019002904', '06019002905', '06019002906', '06019003001', '06019003003', '06019003004', '06019003102', '06019003103', '06019003104', '06019003201', '06019003202', '06019003301', '06019003302', '06019003401', '06019

## Import, inspect and return

Download all three linked artifacts (or right-click in Lab's `exports/` folder → **Download**). Open [RustGIS Web](https://web.rustgis.app/) or [GeoLibre Web](https://web.geolibre.app/) separately, use **Add Data / file picker**, then inspect count, extent and a known `_atlas_source_id`/value/unit/source. The helper adds this primitive string from the retained GeoJSON `Feature.id`; compare it (or the original `GEOID`) across apps. App-generated internal IDs may differ. Compare the same population legend in both; a selected-tract sum remains a whole-tract sum. Record missing fields and keep the original files. Export changes under a new name, then explicitly upload into Lab for further review. The tabs share no kernel state or files.

The generated `.review.md` contains a prompt grounded in this actual package's schema/summary, an intentionally flawed claim and a human checklist. It is visible text, not a model reply. Review manually or use the [optional assistant guidance](browser-gis.ipynb); no inference is required. Ask a prospective map reader which decision it supports, revise one legend/annotation, and retest. Preserve rights for source data, software, model weights, basemaps and your own outputs separately.

Subject continuations: [OneHealth join](courses/onehealth/07_eo_population_joins.ipynb), [Population exports/oracle](courses/population-data-comparison/03_real_exports.ipynb), [RustGIS bridge](courses/rustgis/03_rustgis_bridge.ipynb) and [synthetic fitted DBSCAN](courses/geolibre-geoai/22_rustgis_geoai_bridge.ipynb). Their inputs/runtime/pins remain separate; the shared exporter does not decode rasters, fit these models or certify native branches.


## Optional: prepare an existing geographic CSV

This optional recipe does not run with Run All. Copy its code into a new cell only for the selected subject, with reviewed source-specific metadata. Keep the Census default unchanged. It uses USGS `longitude/latitude/id`, historical cholera **deaths and pumps separately** `LON/LAT/FID`, and the old edition's teaching grid `lon/lat/cell_id` (never planar `x/y`). Raw CSV columns/values survive alongside explicit prefixed numeric derivatives. No observations, coordinates or crosswalks are invented.

Read the retained [USGS/snapshot notices](https://github.com/jltobias/JupyterLite-Learning-Atlas/blob/main/content/courses/data-visualization/repository_examples/data/NOTICES.md), [historical bundle license](https://github.com/jltobias/JupyterLite-Learning-Atlas/blob/main/content/courses/cholera-1854-basic/notices/LICENSE) and [old-edition teaching description](https://github.com/jltobias/JupyterLite-Learning-Atlas/blob/main/content/courses/geospatial-analyses-and-maps-old/notices/README.md). Author a metadata JSON from those actual notices/provider records: source URL, attribution, applicable license statement/evidence, observation time, limitations and units. Unknown rights stay unknown; Census terms are unsuitable. The recipe retains notice text, source CSV hash, selected IDs and mapping. Conversion does not establish rights, geographic accuracy, disease risk or causation.

```python
import csv
from copy import deepcopy
import hashlib
import io
import json
from pathlib import Path
from atlas_gis_handoff import point_feature

csv_profiles = {
    'usgs': {'path': 'courses/data-visualization/repository_examples/data/usgs/events.csv',
        'xyid': ('longitude', 'latitude', 'id'), 'numeric': {'depth_km': 'km', 'magnitude': 'reported magnitude; not a linear count'},
        'status': 'published USGS earthquake snapshot',
        'notice': 'courses/data-visualization/repository_examples/data/NOTICES.md'},
    'cholera-deaths': {'path': 'courses/cholera-1854-basic/resources/cholera_deaths.csv',
        'xyid': ('LON', 'LAT', 'FID'), 'numeric': {'DEATHS': 'recorded deaths at location; no at-risk denominator'},
        'status': 'historical 1854 death-location transcription',
        'notice': 'courses/cholera-1854-basic/notices/LICENSE'},
    'cholera-pumps': {'path': 'courses/cholera-1854-basic/resources/pumps.csv',
        'xyid': ('LON', 'LAT', 'FID'), 'numeric': {},
        'status': 'historical pump-location transcription',
        'notice': 'courses/cholera-1854-basic/notices/LICENSE'},
    'landcover': {'path': 'courses/geospatial-analyses-and-maps-old/data/landcover_grid.csv',
        'xyid': ('lon', 'lat', 'cell_id'), 'numeric': {'elevation_m': 'm; teaching value', 'ndvi': 'unitless teaching value'},
        'status': 'synthetic geographic teaching grid',
        'notice': 'courses/geospatial-analyses-and-maps-old/notices/README.md'},
}

def _prepare_csv_points(kind, supplied_metadata, limit=5):
    profile = csv_profiles[kind]
    assert type(limit) is int and 0 <= limit <= 20, 'Choose an explicit small row limit.'
    assert supplied_metadata['dataset_id'] != 'fresno-census-2020-tracts', 'Supply CSV source metadata, not Census terms.'
    assert supplied_metadata['observation_status'] == profile['status'], 'Retain the historical/teaching/observed status.'
    assert any(profile['notice'] in source['license_evidence'] for source in supplied_metadata['sources']), 'Review and cite the retained notice for this CSV.'
    raw = Path(profile['path']).read_bytes()
    rows = list(csv.DictReader(io.StringIO(raw.decode('utf-8-sig'))))
    x, y, identity = profile['xyid']
    selected = rows[:limit]
    features = []
    for row in selected:
        typed = {'_csv_longitude': float(row[x]), '_csv_latitude': float(row[y])}
        typed.update({'_csv_' + field: float(row[field]) if row[field].strip() else None for field in profile['numeric']})
        assert not set(typed).intersection(row), 'Preserve existing columns; typed field collision.'
        features.append(point_feature(row[identity], typed['_csv_longitude'], typed['_csv_latitude'], {**row, **typed}))
    result_metadata = deepcopy(supplied_metadata)
    result_metadata['field_units'] = {**result_metadata['field_units'], '_csv_longitude': 'degrees longitude',
        '_csv_latitude': 'degrees latitude', **{'_csv_' + field: unit for field, unit in profile['numeric'].items()}}
    result_metadata['csv_preparation'] = {'source_path': profile['path'], 'source_csv_sha256': hashlib.sha256(raw).hexdigest(),
        'input_rows': len(rows), 'selected_ids': [row[identity] for row in selected],
        'selection': 'first explicitly limited source rows, in source order', 'longitude_latitude_id_columns': [x, y, identity],
        'raw_columns_retained': list(rows[0]) if rows else [], 'typed_fields': result_metadata['field_units'],
        'notice_path': profile['notice'], 'retained_notice': Path(profile['notice']).read_text(encoding='utf-8')}
    result_metadata['method'] += ' CSV preparation retained raw strings and explicit typed derivatives; empty numeric strings become null; no reprojection or inference.'
    return {'type': 'FeatureCollection', 'features': features}, result_metadata

# After authoring my-usgs.metadata.json with actual source terms:
# supplied = json.loads(Path('my-usgs.metadata.json').read_text())
# prepared, csv_metadata = _prepare_csv_points('usgs', supplied, limit=5)
# with Path('usgs-prepared.geojson').open('x', encoding='utf-8') as stream:
#     json.dump(prepared, stream, ensure_ascii=False, allow_nan=False)
# metadata = csv_metadata
# input_path = Path('usgs-prepared.geojson')
# collection, input_record = load_input(input_path, metadata)
# Then explicitly choose its typed measurement, threshold, IDs and a new export stem.
```


## Review an uploaded GIS return separately

An app-returned Atlas export is a **review artifact**, not a fresh input to `load_input`: it already contains protected `_atlas_source_id`/`_atlas_handoff` fields, which the loader intentionally rejects. Keep those fields and the original package. Do not strip provenance, restore app IDs automatically, or rerun the fresh-input path. Upload the returned GeoJSON under a new name and inspect it against original portable IDs/values using this optional cell. It reports differences without merging or re-ingesting anything.

```python
import json
from pathlib import Path

def _inspect_returned(returned, original_package, fields):
    assert returned.get('type') == 'FeatureCollection' and isinstance(returned.get('features'), list)
    original = {f['properties']['_atlas_source_id']: f for f in original_package['collection']['features']}
    seen, comparisons = set(), []
    for feature in returned['features']:
        properties = feature.get('properties', {})
        source_id = properties.get('_atlas_source_id')
        assert isinstance(source_id, str) and source_id and source_id not in seen, 'Missing/duplicate portable source ID: review manually.'
        seen.add(source_id)
        previous = original.get(source_id)
        comparisons.append({'source_id': source_id, 'returned_app_id': feature.get('id'),
            'status': 'matched' if previous else 'unmatched; review manually',
            'original_feature_id': previous['id'] if previous else None,
            'values': {field: {'original_present': field in previous['properties'] if previous else False,
                'original': previous['properties'].get(field) if previous else None,
                'returned_present': field in properties, 'returned': properties.get(field)} for field in fields}})
    return comparisons

# After uploading returned-gis.geojson, inspect against this run's original package:
# returned = json.loads(Path('returned-gis.geojson').read_text(encoding='utf-8'))
# comparisons = _inspect_returned(returned, package, list(metadata['field_units']))
# print(json.dumps(comparisons, indent=2))
```

Compare counts, unmatched IDs, changed/missing values, geometry, units and notices before making a new decision. App internal `FeatureID` may differ from GeoJSON `Feature.id`; neither replaces the portable source field. If that field is lost, stop automated comparison and inspect source evidence. A deliberately prepared new analytical input needs a separately documented source/provenance workflow; this recipe provides no automatic conversion back into the exporter.
